# Assignment 3 · Parte 2 – Mapas del Perú

**Grupo 11** · Diplomado PUCP · Fundamentos de Python

**Pregunta:** ¿En qué zonas del país se concentraron las declaratorias de emergencia por lluvias (1 ene – 31 may 2024) y coinciden con las zonas donde más llovió?

## 1. Cargar los datos y las fronteras

In [5]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

# dtype={"ubigeo": str} evita que '01' se convierta en el número 1
dataset = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})

# Fronteras de los 25 departamentos. El código del departamento viene en la columna FIRST_IDDP
departamentos = gpd.read_file("datos/departamentos.geojson")
departamentos = departamentos.rename(columns={"FIRST_IDDP": "ubigeo"})[["ubigeo", "NOMBDEP", "geometry"]]

print("Sistema de coordenadas:", departamentos.crs)
print("Departamentos en el mapa:", departamentos.shape[0])
departamentos.head(3)

Sistema de coordenadas: EPSG:4326
Departamentos en el mapa: 25


,ubigeo,NOMBDEP,geometry
0,01,AMAZONAS,"POLYGON ((-77.75893 -6.96451, -77.84586 -6.976..."
1,02,ANCASH,"POLYGON ((-77.31749 -8.53015, -77.28903 -8.589..."
2,03,APURIMAC,"POLYGON ((-72.47177 -14.6614, -72.57725 -14.68..."


**Sobre el archivo de fronteras:** usamos `datos/departamentos.geojson` (25 departamentos, columna `FIRST_IDDP` con el ubigeo de 2 dígitos), tomado del repositorio `d2cml-ai/Data-Science-Python`. El de la sesión 7 solo trae el nombre del departamento, sin ubigeo, y no permite cruzar por código.

## 2. Cruce por ubigeo (como texto)

Antes de unir, verificamos que el ubigeo esté escrito igual en los dos lados: texto de 2 dígitos con el cero a la izquierda.

In [6]:
print("Mapa  :", departamentos["ubigeo"].dtype, "| ejemplo:", departamentos["ubigeo"].iloc[0])
print("Tabla :", dataset["ubigeo"].dtype, "| ejemplo:", dataset["ubigeo"].iloc[0])
print("Coinciden:", len(set(departamentos["ubigeo"]) & set(dataset["ubigeo"])), "de", len(dataset))

Mapa  : str | ejemplo: 01
Tabla : str | ejemplo: 01
Coinciden: 25 de 25


In [7]:
# how="left" desde el mapa para no perder ningún departamento.
# indicator=True agrega la columna _merge, que dice si cada fila encontró pareja.
mapa = departamentos.merge(dataset, on="ubigeo", how="left", indicator=True)

# Reporte de no cruces
sin_datos = mapa.loc[mapa["_merge"] == "left_only", "NOMBDEP"].tolist()
sin_mapa = sorted(set(dataset["ubigeo"]) - set(departamentos["ubigeo"]))
print("Departamentos del mapa sin datos:", sin_datos)
print("Ubigeos de la tabla que no están en el mapa:", sin_mapa)

# Comprobación: el total no debe cambiar después del cruce
print("Declaratorias en el mapa :", int(mapa["declaratorias"].sum()))
print("Declaratorias en la tabla:", int(dataset["declaratorias"].sum()))

mapa = mapa.drop(columns="_merge")

Departamentos del mapa sin datos: []
Ubigeos de la tabla que no están en el mapa: []
Declaratorias en el mapa : 100
Declaratorias en la tabla: 100


**Reporte del cruce:** cruzaron los 25 departamentos de 25. No quedó ninguno sin datos ni ningún ubigeo sin mapa, y el total de declaratorias (100) es el mismo antes y después del cruce.